# 📝 그래프 데이터 과학 과제 LV2(응용): 이종 투영·중심성 해석·범위 필터

> LV1 에서 익힌 것을 **조합**합니다. 문제는 세 갈래로 묶여 있고, 마지막 문제는 **서술형**입니다.
>
> - **1. 이종 투영과 중심성 해석**: 노드 종류가 둘인 투영·PageRank 대 차수·pandas 로 가공
> - **2. 개인화 PageRank 와 범위 필터**: `sourceNodes`·`nodeLabels` 로 계산 범위 좁히기
> - **3. 조건을 건 투영과 갱신**: 조건을 걸어 일부만 담기·원본이 바뀌면 투영 다시 만들기·이 조각으로 답할 수 있는 것 정리

## 풀이 방법
1. 맨 위 **준비 셀**을 위에서부터 실행하세요.
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).
3. 서술형 문제는 markdown 셀에 직접 써 넣고, 정답 노트북의 모범 서술과 비교하세요.

- 데이터: 의료 지식 그래프의 **질병-유전자 축**입니다. `Disease` 와 `Gene` 이 `ASSOCIATES`(문헌에 보고된 연관)로 이어져 있습니다.
- 규모: 질병 136개, 유전자 13,113개, 연관 12,623건. 질병 하나에 유전자가 수백 개씩 달린 **한쪽으로 쏠린 구조**라는 점을 기억하세요.
- 반드시 **실습 전용 DB**에 연결하세요.

화이팅!

아래 준비 셀들을 위에서부터 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
# 1) 메모리에 올라온 투영부터 내린다. 투영은 이름이 겹치면 다시 못 만들어 재실행이 막힌다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName",
               name=_g["graphName"])

# 2) 저장된 그래프를 지운다. DETACH: 노드에 붙은 관계까지 함께 지운다
run_cypher("MATCH (n) DETACH DELETE n")

print("초기화 완료:", NEO4J_URI,
      "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 남은 투영:", len(run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")))

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# data/hetionet_*.csv 는 Hetionet v1.0 에서 재배포 가능한 출처(CC0/CC BY)만 골라 낸 조각입니다.
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
# 관계 타입마다 양끝 레이블이 정해져 있습니다. 적재할 때 이 표로 레이블을 찍어 줘야
# MATCH 가 인덱스를 타고, 그래야 9만 건이 몇 초 안에 들어갑니다.
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) 레이블마다 id 인덱스를 먼저 만든다. 관계를 붙일 때 이 인덱스로 노드를 찾는다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 csv 를 읽어 레이블별로 나눠 담는다. 레이블마다 CREATE 쿼리가 달라서 미리 갈라 둔다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    # 초기화 직후라 같은 노드가 있을 수 없다. MERGE 대신 CREATE 가 훨씬 빠르다
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 csv 도 타입별로 나눠 담는다
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]   # 이 타입의 출발·도착 레이블을 위 표에서 꺼낸다
    # 2만 건씩 끊어 보낸다. 9만 건을 한 트랜잭션에 넣으면 메모리가 크게 뛴다
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 이번 과제가 쓰는 조각을 훑어봅니다.

In [ ]:
# [제공 코드] 질병-유전자 축의 모양을 먼저 확인합니다
print("질병:", run_cypher("MATCH (d:Disease) RETURN count(d) AS c")[0]["c"],
      "/ 유전자:", run_cypher("MATCH (g:Gene) RETURN count(g) AS c")[0]["c"],
      "/ 연관:", run_cypher("MATCH ()-[r:ASSOCIATES]->() RETURN count(r) AS c")[0]["c"])
# 질병 하나를 예로 유전자가 몇 개나 붙는지 본다(1-2·1-3 이 찾는 1위와는 다른 질병을 골랐다.
# 순위표를 보여 주면 답을 미리 알려 주는 셈이라, 예시는 딱 하나만 이름으로 짚는다)
one_disease = run_cypher("""
    MATCH (d:Disease {name: 'kidney cancer'})-[:ASSOCIATES]->(g:Gene)
    RETURN count(g) AS genes
""")[0]['genes']
print(f"kidney cancer 에 붙은 유전자: {one_disease}개")
# 반대로 유전자 하나를 예로 질병이 몇 개나 붙는지도 본다(역시 1-3 정답과는 다른 유전자다)
one_gene = run_cypher("""
    MATCH (d:Disease)-[:ASSOCIATES]->(g:Gene {name: 'EGFR'})
    RETURN count(d) AS diseases
""")[0]['diseases']
print(f"EGFR 에 붙은 질병: {one_gene}개")
print('→ 질병 하나에 유전자 수백 개, 유전자 하나에 질병 수십 개. 양쪽 다 한쪽으로 쏠린 구조다')

---
# 1. 이종 투영과 중심성 해석

노드 종류가 둘인 투영을 만들고, 두 잣대의 1위가 갈리는 이유를 해석합니다(교안_01 2·3절 · 교안_02 1·5절).

## 1-1. 노드 종류가 둘인 그래프 투영하기
**배경**: `ASSOCIATES` 는 질병에서 유전자로 갑니다. 양끝 레이블이 다르니 **둘 다** 담아야 합니다.

**요구사항**:
- `Disease` 와 `Gene` 노드, `ASSOCIATES` 관계를 **방향을 지워서** **`diseaseGeneGraph`** 로 투영하세요.
- 투영 결과의 `nodeCount` 를 **`n_nodes`**, `relationshipCount` 를 **`n_rels`** 에 담으세요.
- 원본 관계 건수를 Cypher 로 세어 **`raw_rels`** 에 담고, 투영이 그 2배인지 확인하세요.

**예시**: `n_nodes` 는 13,249, `raw_rels` 는 12,623, `n_rels` 는 25,246 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 레이블은 두 개짜리 리스트, 관계는 방향을 지우는 설정 맵으로 넘긴다.
- 만든 뒤에는 반드시 원본 건수와 대조한다. 이것이 이 단원의 검산 습관이다.

세부구현:
1. 투영을 만들며 노드 수와 관계 수를 함께 YIELD 로 받는다.
2. 받은 첫 행에서 두 값을 각각 꺼내 변수에 담는다.
3. 평범한 MATCH 로 원본 관계를 세어 raw_rels 에 담는다.
4. 세 값을 함께 출력해 2배 관계가 성립하는지 눈으로 확인한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_nodes == 13249, '노드 수가 다릅니다. Disease 와 Gene 을 모두 담았는지 확인하세요'
assert raw_rels == 12623, \
    ('원본 관계 수가 다릅니다. ASSOCIATES 만 셌는지 확인하세요. 아래 3-2 를 이미 실행했다면 '
     'asthma 의 관계 222건이 지워진 상태라 값이 다릅니다. 그때는 맨 위 제공 코드 '
     '셀부터 다시 실행하세요')
assert n_rels == raw_rels * 2, ('무방향이면 투영 관계 수가 원본의 2배여야 합니다. orientation 설정을 '
                                '확인하세요. 고쳐 다시 만들려면 맨 위 제공 코드 셀 셋(연결·초기화·적재)을 '
                                '위에서부터 다시 실행한 뒤 이 셀을 다시 실행하세요')
# 세 값을 손으로 적어도 통과하지 않게, 채점이 그 자리에서 다시 세어 대조한다
_live = run_cypher("CALL gds.graph.list('diseaseGeneGraph') "
                   "YIELD nodeCount, relationshipCount "
                   "RETURN nodeCount, relationshipCount")
assert _live, 'diseaseGeneGraph 투영이 없습니다. 투영을 실제로 만들었는지 확인하세요'
assert n_nodes == _live[0]['nodeCount'] and n_rels == _live[0]['relationshipCount'], \
    '카탈로그에서 다시 읽은 값과 다릅니다. YIELD 로 받은 값을 그대로 담았는지 확인하세요'
_raw = run_cypher("MATCH ()-[r:ASSOCIATES]->() RETURN count(r) AS c")[0]['c']
assert raw_rels == _raw, '원본 관계 수를 직접 세었는지 확인하세요(지금 다시 센 값과 다릅니다)'
print('✅ 통과!')

## 1-2. PageRank 1위와 차수 1위를 함께 뽑고 해석하기
**배경**: 같은 투영에 두 가지 잣대를 대 봅니다. 답이 같을까요, 다를까요?

**요구사항**:
- **`diseaseGeneGraph`** 에서 `gds.pageRank.stream` 으로 1위 노드 이름을 **`pr_top`** 에 담으세요.
- 같은 투영에서 `gds.degree.stream` 으로 1위 노드 이름을 **`deg_top`** 에 담으세요.
- 두 이름이 다른지 판정해 **`differ`** 에 참거짓으로 담으세요.
- 아래 markdown 셀에 **두 답이 왜 갈렸는지** 한두 문장으로 쓰세요.

**예시**: `pr_top` 은 `'IgA glomerulonephritis'`, `deg_top` 은 `'hematologic cancer'`, `differ` 는 `True` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 알고리즘의 호출 모양이 같다. 이름만 바꿔 두 번 부르면 된다.
- 같은 코드를 두 번 쓰기 싫다면 알고리즘 이름을 인자로 받는 함수를 만들어도 좋다.

세부구현:
1. stream 으로 nodeId 와 score 를 받아 노드 이름을 꺼낸다.
2. 점수 내림차순으로 정렬해 한 건만 받는다.
3. 알고리즘 이름만 바꿔 한 번 더 한다.
4. 두 이름을 비교한 결과를 differ 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert pr_top == 'IgA glomerulonephritis', 'PageRank 1위 이름이 다릅니다. diseaseGeneGraph 로 실행했는지 확인하세요'
assert deg_top == 'hematologic cancer', '차수 1위 이름이 다릅니다. gds.degree.stream 을 같은 투영으로 실행했는지 확인하세요'
assert differ is True, '두 1위는 서로 다른 노드여야 합니다. 각 변수에 이름 문자열을 담았는지 확인하세요'
# 지문이 두 이름을 알려 주므로, 채점이 그 자리에서 둘 다 다시 돌려 대조한다
_tail = ("YIELD nodeId, score RETURN gds.util.asNode(nodeId).name AS name "
         "ORDER BY score DESC, name LIMIT 1")
_pr = run_cypher(f"CALL gds.pageRank.stream('diseaseGeneGraph') {_tail}")[0]['name']
_dg = run_cypher(f"CALL gds.degree.stream('diseaseGeneGraph') {_tail}")[0]['name']
assert pr_top == _pr, '지금 다시 계산한 PageRank 1위와 다릅니다. 직접 실행했는지 확인하세요'
assert deg_top == _dg, '지금 다시 잰 차수 1위와 다릅니다. 직접 실행했는지 확인하세요'
print('✅ 통과!')

**해석**: 두 답이 갈린 이유를 한두 문장으로 쓰세요.

*(여기에 자신의 해석을 서술하세요)*

## 1-3. 결과를 pandas 로 가공해 유전자만 보기
**배경**: 1-2 의 상위권은 전부 질병이었습니다. "**유전자 중에서는** 무엇이 중심인가"를 물으려면 결과를 걸러야 합니다.

**요구사항**:
- 이 문제부터 pandas 를 씁니다. 준비 셀은 pandas 를 올리지 않으니 답안 셀 맨 위에 `import pandas as pd` 를 먼저 적으세요.
- `gds.pageRank.stream` 결과를 노드 이름·**노드 종류**·점수 세 컬럼으로 받으세요. 노드 종류는 `labels(gds.util.asNode(nodeId))[0]` 로 꺼냅니다.
- 그 결과로 pandas DataFrame **`rank_df`** 를 만드세요. 컬럼은 **이 순서로** `name`·`kind`·`score` 입니다.
- 결과는 **전체 행**을 받으세요(`LIMIT` 을 걸지 마세요). 유전자만 고르는 일은 pandas 쪽에서 합니다.
- `kind` 가 `'Gene'` 인 행만 골라 점수가 가장 높은 유전자 이름을 **`top_gene`** 에 담으세요.

**예시**: `top_gene` 은 `'TP53'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- run_cypher 는 dict 리스트를 주므로 DataFrame 생성자에 그대로 넣으면 키가 컬럼이 된다.
- 정렬은 Cypher 쪽에서 미리 해 두면 pandas 에서 다시 정렬할 필요가 없다.

세부구현:
1. stream 결과에서 이름·노드 종류·점수를 RETURN 하고 점수 내림차순으로 정렬한다.
2. 받은 리스트로 DataFrame 을 만든다.
3. 노드 종류가 유전자인 행만 남긴다.
4. 이미 정렬돼 있으므로 맨 앞 행의 이름을 꺼낸다. 위치로 고르는 편이 헷갈리지 않는다.
```

</details>

In [ ]:
import pandas as pd
res = run_cypher('''
CALL gds.pageRank.stream('diseaseGeneGraph')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS kind, score
ORDER BY score DESC, name ASC
''')
rank_df = pd.DataFrame(res)
gene_df = rank_df[rank_df['kind'] == 'Gene']
top_gene = gene_df.iloc[0]['name']
print(f'top_gene: {top_gene}')


In [ ]:
# [자가채점]
assert list(rank_df.columns) == ['name', 'kind', 'score'], "rank_df 의 컬럼은 name, kind, score 순서여야 합니다. RETURN 절의 별칭을 확인하세요"
assert len(rank_df) == 13249, '투영에 담긴 노드 전부가 결과에 있어야 합니다. LIMIT 을 걸지 않았는지 확인하세요'
assert top_gene == 'TP53', '유전자 중 1위 이름이 다릅니다. kind 가 Gene 인 행만 걸렀는지, 점수 내림차순인지 확인하세요'
# 이름만 적어도 통과하지 않게, 같은 순위를 Cypher 쪽에서 한 번 더 뽑아 대조한다
_gene = run_cypher(
    "CALL gds.pageRank.stream('diseaseGeneGraph') YIELD nodeId, score "
    "WITH gds.util.asNode(nodeId) AS n, score WHERE 'Gene' IN labels(n) "
    "RETURN n.name AS name ORDER BY score DESC, name LIMIT 1")[0]['name']
assert top_gene == _gene, \
    '지금 다시 계산한 유전자 1위와 다릅니다. rank_df 에서 걸러 냈는지 확인하세요'
print('✅ 통과!')

## 1-4. 세 번째 잣대: 매개 중심성
**배경**: 차수와 PageRank 는 둘 다 "연결이 몰린 곳"을 찾습니다. **매개 중심성**은 다른 것을 찾습니다. **모든 노드 쌍의 최단 경로 중 몇 개가 나를 지나가는가**, 곧 **길목**입니다.

**요구사항**:
- `diseaseGeneGraph` 에 `gds.betweenness.stream` 을 돌려 1위 이름을 **`bc_top`** 에 담으세요.
- 그 1위의 **점수(`score`)** 도 변수 **`bc_score`** 에 담으세요. 이 값은 지문에 없습니다. 자가채점이 지금 계산한 값과 대조합니다.
- 그 1위가 **차수 1위(`deg_top`)와 같은지** 판정해 **`bc_equals_deg`** 에 참거짓으로 담으세요.
  (1-2 에서 담은 `deg_top` 을 그대로 씁니다.)

**예시**: `bc_top` 은 `'hematologic cancer'`, `bc_equals_deg` 는 `True` 입니다. `bc_score` 는 백만 단위의 큰 수입니다. 세 잣대가 **늘 다른 답을 내는 것은 아닙니다.** 이 조각에서는 매개와 차수의 1위가 같고, PageRank 만 다른 답을 냅니다.

> **방향을 지운 투영이어야 합니다.** 최단 경로를 세는 알고리즘이라 화살표를 거슬러 못 가면 경로가 거의 안 생깁니다. 1-1 에서 무방향으로 담아 두었으니 그대로 쓰면 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 호출 모양은 pageRank·degree 와 똑같다. 프로시저 이름만 바꾼다.
- 점수는 개수가 아니라 '지나간 경로 수' 라 자릿수가 크다. 순위만 읽으면 된다.

세부구현:
1. gds.betweenness.stream 을 돌려 score 내림차순·이름 오름차순 1건을 받는다.
2. 그 행에서 이름을 bc_top 에, 점수를 bc_score 에 담는다.
3. bc_top 과 deg_top 이 같은지 비교해 bc_equals_deg 에 담는다.
```

</details>

In [ ]:
bc_top = run_cypher('''
CALL gds.betweenness.stream('diseaseGeneGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC LIMIT 1
''')[0]['name']
bc_equals_deg = (bc_top == deg_top)
print(f'bc_top: {bc_top}, bc_equals_deg: {bc_equals_deg}')


In [ ]:
# [자가채점]
assert bc_top == 'hematologic cancer', \
    '매개 중심성 1위 이름이 다릅니다. diseaseGeneGraph 로 실행했는지, score 내림차순인지 확인하세요'
assert bc_equals_deg is True, \
    '이 조각에서는 매개 1위와 차수 1위가 같습니다. deg_top 을 1-2 에서 제대로 담았는지 확인하세요'
# 이름만 적어도 통과하지 않게, 채점이 그 자리에서 다시 돌려 이름과 점수를 함께 대조한다
_bc = run_cypher("CALL gds.betweenness.stream('diseaseGeneGraph') YIELD nodeId, score "
                 "RETURN gds.util.asNode(nodeId).name AS name, score "
                 "ORDER BY score DESC, name LIMIT 1")[0]
_bc_name, _bc_score = _bc['name'], _bc['score']
assert bc_top == _bc_name, \
    '지금 다시 계산한 매개 1위와 다릅니다. gds.betweenness.stream 을 직접 실행했는지 확인하세요'
# 점수는 지문에 없다. 병렬로 합산하는 순서가 매번 달라 끝자리가 조금 흔들리므로 0.1% 오차를 허용한다
assert abs(bc_score - _bc_score) <= _bc_score * 1e-3, \
    f'1위 점수가 지금 계산한 값({_bc_score:,.0f})과 다릅니다. gds.betweenness.stream 을 직접 실행해 1위 행의 score 를 담았는지 확인하세요'
print('✅ 통과!')

---
# 2. 개인화 PageRank 와 범위 필터

출발점을 정한 순위를 뽑고, 계산 범위를 좁히면 답이 어떻게 달라지는지 봅니다(교안_02 3·4절).

## 2-1. 한 질병 관점의 개인화 PageRank
**배경**: 전체 순위 말고 **특정 질병과 가까운 것**을 묻습니다.

**요구사항**:
- **`diseaseGeneGraph`** 에서 질병 **`'type 2 diabetes mellitus'`**(2형 당뇨병)을 출발점으로 개인화 PageRank 를 실행하세요. 출발 질병 이름은 **`$name` 파라미터**로 넘기고, 쿼리 문자열은 **`ppr_query`** 에 담으세요.
- `RETURN` 절에 이름 별칭 **`name`** 과 노드 종류 별칭 **`kind`** 를 두세요.
- 상위 3개를 자르는 일까지 **쿼리 안에서** 하세요(`ORDER BY score DESC LIMIT 3`).
- 그렇게 받은 이름 리스트를 **`ppr_top3`** 에 담으세요.
- 그중 1위를 뺀 2·3위의 `kind` 가 모두 **`Disease`** 인지 판정해 **`all_disease`** 에 참거짓으로 담으세요.

**예시**: 이 출발점에서는 1위가 출발점 자신이라 `ppr_top3[0]` 은 `'type 2 diabetes mellitus'` 입니다(출발점이 점수를 나눠 줄 이웃이 적으면 이웃이 앞설 수도 있습니다). 2·3위는 당뇨병과 연관 유전자를 많이 공유하는 질병 이름이 나오고, `all_disease` 는 `True` 입니다 (2·3위 값은 자가채점에서 확인하세요).

> 채점 셀은 같은 `ppr_query` 를 **다른 질병 이름**으로 한 번 더 실행합니다. `ppr_top3` 에 값을 직접 적어 넣으면 여기서 걸립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 출발 노드를 MATCH 로 찾아 리스트로 모은 뒤 알고리즘 설정에 넘긴다.
- 출발 이름을 파라미터로 빼 두면 같은 쿼리를 다른 질병에도 그대로 쓸 수 있다.
- 2·3위가 질병인지 보려면 노드 종류도 함께 받아야 한다.

세부구현:
1. 질병 레이블과 이름으로 출발 노드를 찾되, 이름 자리에 $name 파라미터를 쓴다.
2. 그 노드를 리스트로 모아 sourceNodes 에 넘긴다.
3. 이름과 노드 종류를 별칭까지 맞춰 받고, 점수 내림차순 3건으로 쿼리 안에서 자른다.
4. 그 쿼리 문자열을 ppr_query 에 담고, 출발 질병 이름은 run_cypher 의 name 인자로 넘긴다.
5. 이름만 뽑아 리스트로 만들고, 두 번째와 세 번째 행의 노드 종류가 모두 질병인지 확인한다.
```

</details>

In [ ]:
res = run_cypher('''
MATCH (d:Disease {name: 'type 2 diabetes mellitus'})
WITH collect(d) AS sources
CALL gds.pageRank.stream('diseaseGeneGraph', {
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
RETURN n.name AS name, labels(n)[0] AS kind
ORDER BY score DESC, name ASC
LIMIT 3
''')
ppr_top3 = [row['name'] for row in res]
all_disease = all(row['kind'] == 'Disease' for row in res)
print(f'ppr_top3: {ppr_top3}, all_disease: {all_disease}')


In [ ]:
# [자가채점]
assert ppr_top3[0] == 'type 2 diabetes mellitus', '이 출발점에서는 1위가 출발점 자신입니다. sourceNodes 에 넘긴 노드를 확인하세요'
assert ppr_top3[1] == 'hypertension', '2위 이름이 다릅니다. diseaseGeneGraph 로 실행했는지 확인하세요'
assert ppr_top3[2] == 'obesity', '3위 이름이 다릅니다. LIMIT 3 으로 세 건을 받았는지 확인하세요'
assert all_disease is True, '2·3위의 노드 종류가 모두 Disease 여야 합니다. labels 로 종류를 함께 받았는지 확인하세요'
# 세 이름이 지문에 있으므로, 채점이 같은 개인화 PageRank 를 다시 돌려 대조한다
_ppr = [r['name'] for r in run_cypher(
    "MATCH (s:Disease {name: $name}) WITH collect(s) AS src "
    "CALL gds.pageRank.stream('diseaseGeneGraph', {sourceNodes: src}) "
    "YIELD nodeId, score "
    "RETURN gds.util.asNode(nodeId).name AS name "
    "ORDER BY score DESC, name LIMIT 3", name='type 2 diabetes mellitus')]
assert ppr_top3 == _ppr, \
    '지금 다시 계산한 상위 3개와 다릅니다. sourceNodes 로 직접 실행했는지 확인하세요'
# 값을 손으로 적어도 통과하지 않게, 같은 쿼리를 다른 질병으로 한 번 더 돌린다
other = run_cypher(ppr_query, name='breast cancer')
assert other and 'name' in other[0], 'ppr_query 의 RETURN 별칭을 name 으로 두세요'
_other_names = [row['name'] for row in other]
_shown = _other_names[:5]
assert _other_names == ['breast cancer', 'prostate cancer', 'hematologic cancer'], \
    f'같은 쿼리를 다른 질병으로 돌리면 그 질병과 연관 유전자를 많이 공유하는 질병이 나와야 합니다(현재 {_shown}). ppr_query 에 쿼리 문자열을 담고 출발 이름을 $name 으로 넘겼는지, LIMIT 3 이 쿼리 안에 있는지 확인하세요'
print('✅ 통과!')

## 2-2. 계산 범위를 좁히면 무슨 일이 벌어지나
**배경**: `nodeLabels` 는 결과를 걸러내는 것이 아니라 **그래프 자체를 줄입니다.** 1-3 의 "결과 걸러내기"와 무엇이 다른지 직접 확인합니다.

**요구사항**:
- **`diseaseGeneGraph`** 에서 `gds.pageRank.stream` 을 실행하되 설정에 **`nodeLabels`** 를 넣으세요. 레이블 리스트는 **`$labels` 파라미터**로 넘기고(`{nodeLabels: $labels}`), 이 문제에서는 `['Disease']` 를 넘깁니다.
- 쿼리 문자열은 **`filter_query`** 에 담으세요.
- 점수 컬럼의 별칭은 **`score`** 로 두세요.
- 결과 점수를 소수 4자리로 반올림한 **집합**을 만들어 **`score_set`** 에 담으세요 (파이썬 `set` 입니다).
- 결과 행 수를 **`n_rows`** 에 담으세요.

**예시**: `n_rows` 는 136(질병 수), `score_set` 의 원소는 **한 개**입니다. 모든 질병이 똑같은 점수를 받습니다.

> 채점 셀은 같은 `filter_query` 를 `['Gene']` 으로 한 번 더 실행합니다. 값을 직접 적어 넣으면 여기서 걸립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 알고리즘 두 번째 인자로 설정 맵을 넘긴다. 개인화 때 sourceNodes 를 넘기던 그 자리다.
- 레이블 리스트를 파라미터로 빼 두면 같은 쿼리를 다른 레이블에도 그대로 쓸 수 있다.
- 점수를 그대로 집합에 넣으면 부동소수 끝자리 차이로 원소가 여러 개가 될 수 있다. 반올림한다.

세부구현:
1. 설정 맵의 노드 레이블 자리에 $labels 파라미터를 넣어 stream 을 부른다.
2. 그 쿼리 문자열을 filter_query 에 담고, 레이블 리스트는 run_cypher 의 labels 인자로 넘긴다.
3. 점수만 받아 파이썬 리스트로 만든다.
4. 각 점수를 소수 4자리로 반올림해 집합으로 만든다.
5. 행 수와 집합을 출력해 확인한다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.pageRank.stream('diseaseGeneGraph', {
    nodeLabels: ['Disease']
})
YIELD nodeId, score
RETURN count(*) AS n, collect(DISTINCT round(score, 4)) AS s
''')[0]
n_rows = res['n']
score_set = set(res['s'])
print(f'n_rows: {n_rows}, score_set: {score_set}')


In [ ]:
# [자가채점]
assert n_rows == 136, '질병 수만큼 행이 나와야 합니다. nodeLabels 를 Disease 하나로 줬는지 확인하세요'
# 관계가 하나도 안 남으면 모든 점수가 정확히 1 - dampingFactor 다. 그 값 자체를 확인해야
# '소수 4자리로 반올림하라'는 요구가 채점에 실제로 걸린다(반올림을 빼먹으면 부동소수 끝자리가 흩어져
# {0.15, 0.1500000000000002, ...} 처럼 원소가 여러 개로 보인다)
assert score_set == {0.15}, ('관계가 하나도 안 남으면 모든 점수가 1 - dampingFactor = 0.15 입니다. '
                             'nodeLabels 를 Disease 하나로 줬는지, 소수 4자리로 반올림했는지 확인하세요')
# 값을 손으로 적어도 통과하지 않게, 채점이 같은 필터로 다시 돌려 행 수까지 대조한다
_rows = run_cypher(
    "CALL gds.pageRank.stream('diseaseGeneGraph', {nodeLabels: ['Disease']}) "
    "YIELD nodeId, score RETURN count(*) AS n, collect(DISTINCT round(score, 4)) AS s")[0]
assert n_rows == _rows['n'], '지금 다시 돌린 행 수와 다릅니다. 직접 실행했는지 확인하세요'
assert score_set == set(_rows['s']), '지금 다시 돌린 점수 집합과 다릅니다. 직접 실행했는지 확인하세요'
# 같은 filter_query 를 이번엔 유전자로 돌린다. 쿼리를 실제로 담았어야 여기를 통과한다
_g = run_cypher(filter_query, labels=['Gene'])
_g_rows = len(_g)
assert _g_rows == 13113, \
    f'같은 쿼리를 Gene 으로 돌리면 유전자 수 13,113 행이 나와야 합니다(현재 {_g_rows}). filter_query 에 쿼리 문자열을 담고 nodeLabels 를 $labels 로 넘겼는지 확인하세요'
assert 'score' in _g[0], 'filter_query 의 점수 컬럼 별칭을 score 로 두세요'
assert {round(r['score'], 4) for r in _g} == {0.15}, \
    '유전자만 남겨도 관계의 반대쪽 끝인 질병이 빠져 선이 하나도 안 남습니다. 그래서 점수는 역시 0.15 하나여야 합니다'
print('✅ 통과!')

**해석**: 왜 모든 질병의 점수가 같아졌을까요? 한두 문장으로 쓰세요.

*(여기에 자신의 해석을 서술하세요)*

---
# 3. 조건을 건 투영과 갱신

조건을 걸어 일부만 담아 보고, 원본이 바뀌면 투영을 다시 만듭니다. 마지막으로 이 조각으로 답할 수 있는 것과 없는 것을 정리합니다(교안_01 2·5절).

## 3-1. 조건을 걸어 담기: Cypher 투영
**배경**: 지금까지 쓴 `CALL gds.graph.project(이름, 레이블, 관계)` 는 **레이블 전체**를 담습니다. 그래서 `diseaseGeneGraph` 에는 연관 유전자가 한둘뿐인 질병까지 다 들어와 있습니다. **조건을 걸어 일부만 담으려면** 교안_01 5절에서 본 **Cypher 투영**에 무방향 옵션(다섯 번째 인자)을 얹어, 조건에 맞는 짝만 담습니다.

`CALL` 이 아니라 **`RETURN`** 자리에 온다는 점이 다릅니다. `MATCH`·`WHERE` 로 고른 짝만 담깁니다.

**요구사항**:
- 연관 유전자가 **100개 이상**인 질병만 골라, 그 질병과 그 질병의 연관 유전자를 **`bigDiseaseGraph`** 로 **무방향** 투영하세요.
- 네 번째 인자에는 교안_01 처럼 `{sourceNodeLabels: labels(출발), targetNodeLabels: labels(도착)}` 를, 다섯 번째 인자에는 `{undirectedRelationshipTypes: ['*']}` 를 넘기세요.
- 담긴 노드 수를 **`narrow_nodes`**, 관계 수를 **`narrow_rels`** 에 담으세요.

**예시**: `narrow_nodes` 는 `4692`, `narrow_rels` 는 `18892` 입니다. 조건을 만족하는 질병은 **39개**뿐인데 노드가 4,692개인 이유는, 그 39개 질병이 거느린 **유전자까지 함께** 담기기 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 질병마다 연관 유전자 수를 세어 조건에 맞는 질병만 남긴다(WITH 로 집계한 뒤 WHERE).
- 그 질병들의 관계를 다시 MATCH 해 양끝을 project 에 넘긴다.
- 집계 함수라 결과가 한 줄로 오고, nodeCount·relationshipCount 를 담은 맵을 돌려준다.

세부구현:
1. 질병과 유전자를 잇는 관계로 세고, 100개 이상인 질병만 남긴다.
2. 남은 질병으로 다시 MATCH 해 유전자를 잡는다.
3. RETURN 자리에서 project 를 부르며 네 번째·다섯 번째 인자를 요구사항대로 넣는다.
4. 돌려받은 맵에서 노드 수·관계 수를 꺼낸다.
```

</details>

In [ ]:
node_query = '''
MATCH (d:Disease)-[:ASSOCIATES]->(g:Gene)
WITH d, count(g) AS gene_count WHERE gene_count >= 100
RETURN id(d) AS id, ['Disease'] AS labels
UNION
MATCH (d:Disease)-[:ASSOCIATES]->(g:Gene)
WITH d, count(g) AS gene_count WHERE gene_count >= 100
MATCH (d)-[:ASSOCIATES]->(g:Gene)
RETURN DISTINCT id(g) AS id, ['Gene'] AS labels
'''
rel_query = '''
MATCH (d:Disease)-[:ASSOCIATES]->(g:Gene)
WITH d, count(g) AS gene_count WHERE gene_count >= 100
MATCH (d)-[r:ASSOCIATES]->(g:Gene)
RETURN id(d) AS source, id(g) AS target, 'ASSOCIATES' AS type
'''
res = run_cypher('''
CALL gds.graph.project.cypher(
    'bigDiseaseGraph',
    $nodeQuery,
    $relQuery,
    {undirectedRelationshipTypes: ['*']}
)
YIELD nodeCount, relationshipCount
RETURN nodeCount, relationshipCount
''', nodeQuery=node_query, relQuery=rel_query)[0]
narrow_nodes = res['nodeCount']
narrow_rels = res['relationshipCount']
print(f'narrow_nodes: {narrow_nodes}, narrow_rels: {narrow_rels}')


In [ ]:
# [자가채점]
assert narrow_nodes == 4692, \
    ('노드 수가 다릅니다. 조건을 gene_count >= 100 으로 걸었는지 확인하세요. '
     '아래 3-2 를 이미 실행했다면 원본에서 asthma 의 관계가 지워진 상태라 조건을 '
     '만족하는 질병이 하나 줄어 값이 다릅니다. 그때는 맨 위 제공 코드 셀부터 다시 '
     '실행한 뒤 이 문항으로 오세요')
assert narrow_rels == 18892, \
    '관계 수가 다릅니다. undirectedRelationshipTypes 로 무방향으로 담았는지 확인하세요'
# 값만 적어도 통과하지 않게, 투영이 실제로 메모리에 있는지 카탈로그에서 확인한다
_big = run_cypher("CALL gds.graph.list('bigDiseaseGraph') "
                  "YIELD nodeCount, relationshipCount "
                  "RETURN nodeCount, relationshipCount")
assert _big, 'bigDiseaseGraph 투영이 없습니다. 투영을 실제로 만들었는지 확인하세요'
assert _big[0]['nodeCount'] == narrow_nodes, \
    '카탈로그의 노드 수와 담은 값이 다릅니다. project 가 돌려준 값을 그대로 담았는지 확인하세요'
assert _big[0]['relationshipCount'] == narrow_rels, \
    '카탈로그의 관계 수와 담은 값이 다릅니다'
# 전체를 담은 1-1 투영보다 작아야 조건이 실제로 걸린 것이다
assert narrow_nodes < 13249, \
    '조건을 건 투영이 전체 투영보다 작아야 합니다. WHERE 가 걸렸는지 확인하세요'
print('✅ 통과!')

## 3-2. 원본이 바뀌면 투영을 다시 만든다
**배경**: 투영은 **만든 시점의 사본**이라 원본 변경을 따라가지 않습니다. 1-1 에서 만든 `diseaseGeneGraph` 의 원본 관계를 지워 보고 확인합니다. 카탈로그에는 3-1 의 `bigDiseaseGraph` 도 함께 올라와 있으니 이름을 헷갈리지 마세요.

> **이 문제는 원본을 바꿉니다.** 실행한 뒤 1절이나 2절로 되돌아가 다시 풀려면 맨 위 제공 코드 셀부터 다시 실행하세요.

**요구사항**:
- **`diseaseGeneGraph`** 투영의 `relationshipCount` 를 **`old_rels`** 에 담으세요.
- 원본에서 질병 `'asthma'` 에 붙은 `ASSOCIATES` 관계를 **모두 삭제**하세요 (222건입니다).
- 삭제 직후 `diseaseGeneGraph` 의 `relationshipCount` 를 다시 읽어 **`still_rels`** 에 담으세요 (변하지 않아야 합니다).
- 그 투영을 **내리고 같은 이름으로 다시 만든 뒤** `relationshipCount` 를 **`new_rels`** 에 담으세요.

**예시**: `old_rels` 와 `still_rels` 는 25,246, `new_rels` 는 24,802 입니다(222건이 양방향 두 건씩 빠집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 순서가 핵심이다. 원본을 고치기 전에 옛 투영의 관계 수를 먼저 읽어 둔다.
- 같은 이름으로 다시 만들려면 반드시 먼저 지워야 한다.

세부구현:
1. 목록 프로시저에 diseaseGeneGraph 를 주고 관계 수를 읽어 old_rels 에 담는다.
2. 그 질병에서 나가는 연관 관계를 MATCH 로 잡아 DELETE 한다.
3. 관계 수를 다시 읽어 still_rels 에 담는다. 값이 그대로일 것이다.
4. 투영을 지우고 1-1 과 똑같은 설정으로 다시 만든 뒤 관계 수를 new_rels 에 담는다.
```

</details>

In [ ]:
old_rels = run_cypher("CALL gds.graph.list('diseaseGeneGraph') YIELD relationshipCount RETURN relationshipCount")[0]['relationshipCount']
run_cypher("MATCH (:Disease {name: 'asthma'})-[r:ASSOCIATES]->() DELETE r")
still_rels = run_cypher("CALL gds.graph.list('diseaseGeneGraph') YIELD relationshipCount RETURN relationshipCount")[0]['relationshipCount']
run_cypher("CALL gds.graph.drop('diseaseGeneGraph') YIELD graphName")
new_rels = run_cypher('''
CALL gds.graph.project(
    'diseaseGeneGraph',
    ['Disease', 'Gene'],
    {
        ASSOCIATES: {type: 'ASSOCIATES', orientation: 'UNDIRECTED'}
    }
)
YIELD relationshipCount
RETURN relationshipCount
''')[0]['relationshipCount']
print(f'old_rels: {old_rels}, still_rels: {still_rels}, new_rels: {new_rels}')


In [ ]:
# [자가채점]
assert old_rels == 25246, \
    ('삭제 전 관계 수가 25,246 이 아닙니다. 이 셀을 이미 한 번 실행했다면 원본에서 '
     'asthma 의 관계가 지워진 상태라 값이 다릅니다. 그때는 맨 위 제공 코드 셀부터 다시 '
     '실행한 뒤 이 문항으로 오세요. 처음 실행인데도 다르다면 1-1 투영을 먼저 만들었는지 '
     '확인하세요')
assert still_rels == old_rels, '삭제 직후에도 투영의 관계 수는 그대로여야 합니다. 투영을 미리 다시 만들지 않았는지 확인하세요'
# 학생 변수만 보면 재투영을 건너뛰어도 통과한다. 지금 메모리에 있는 투영을 직접 읽는다
_live = run_cypher("CALL gds.graph.list('diseaseGeneGraph') "
                   "YIELD relationshipCount RETURN relationshipCount")
assert _live, 'diseaseGeneGraph 투영이 없습니다. 같은 이름으로 다시 만들었는지 확인하세요'
_live_rels = _live[0]['relationshipCount']
assert _live_rels == 24802, \
    f'지금 메모리에 있는 투영의 관계가 {_live_rels}건입니다. 투영을 내리고 같은 설정으로 다시 만들었는지 확인하세요'
assert new_rels == 24802, '다시 투영한 관계 수가 다릅니다. asthma 의 ASSOCIATES 만 지웠는지, 같은 설정으로 재투영했는지 확인하세요'
left = run_cypher("MATCH (:Disease {name: 'asthma'})-[r:ASSOCIATES]->() RETURN count(r) AS cnt")[0]['cnt']
assert left == 0, 'asthma 의 ASSOCIATES 가 아직 남아 있습니다. DELETE 를 실행했는지 확인하세요'
print('✅ 통과!')

## 3-3. 정리: 이 조각으로 답할 수 있는 질문과 없는 질문 (서술형)
**배경**: 오늘 쓴 `diseaseGeneGraph` 는 질병과 유전자만 담은 조각입니다.

**요구사항**: 아래 두 가지를 각각 한두 문장으로 쓰세요.

1. 이 투영으로 **답할 수 있는** 질문 하나와, 그 답을 얻으려면 어떤 중심성을 쓸지.
2. 이 투영으로 **답할 수 없는** 질문 하나와, 왜 답할 수 없는지. 답하려면 무엇을 더 담아야 하는지.

*(여기에 자신의 해석을 서술하세요)*

---
## 🧹 다 쓴 투영 내리기
투영은 노트북 커널이 아니라 **Neo4j 서버 메모리**에 있습니다. 노트북을 닫아도 사라지지 않고, 같은 데이터베이스에 붙은 다른 사람에게도 보입니다. 오늘 만든 것을 내리고 마칩니다.

In [ ]:
# [제공 코드] 이 과제에서 만든 투영을 내립니다: 이 셀은 실행만 하세요.
# 이름을 하나씩 적습니다. 카탈로그를 통째로 지우면 남이 쓰는 중인 투영까지 함께 내려갑니다
# 두 번째 인자 false 는 "그 이름이 없으면 그냥 넘어가라" 는 뜻입니다
for _name in ['diseaseGeneGraph', 'bigDiseaseGraph']:
    run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
               name=_name)
print('남은 투영:', [row['graphName'] for row in
      run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")])

---
수고했어요! 자가채점이 있는 문제는 모두 **✅ 통과!** 로 끝났는지 확인하세요. 1-3 과 2-2 의 차이(결과를 거르는 것 대 그래프를 줄이는 것)를 말로 설명할 수 있으면 됩니다.